### Fractional Polynomial Approach

In [1]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

In [2]:
SEED = 70
np.random.seed(SEED)
random.seed(SEED)

In [3]:
CROSSOVER_RATE = 0.85
MUTATION_RATE = 0.1

configs = [
    {"id": 1, "pop_size": 20,  "generation": 20},
    {"id": 2, "pop_size": 20,  "generation":100},
    {"id": 3, "pop_size": 50, "generation":20},
    {"id": 4, "pop_size": 50, "generation":100}
]

In [4]:
def load_dataset(path, time_col="T", event_col="E"):
    df = pd.read_csv(path)

    T = df[time_col].values.astype(float)
    E = df[event_col].values.astype(int)

    X = df.drop(columns=[time_col, event_col]).values.astype(float)

    return X, T, E

In [5]:
path_gbsg = "../data/preprocess-data/preprocess_gbsg.csv"
path_hm = "../data/preprocess-data/preprocess_haberman.csv"
path_sd = "../data/preprocess-data/preprocess_simulated_data.csv"

### Fractional Polynomial Approach

Fitness function

Fitness of a solution measure by Integrated Brier Score. Implement IBS Score   https://github.com/square/pysurvival/blob/master/pysurvival/utils/metrics.py

In [6]:
def fractional_polynomial(X, degrees):
    Z = np.zeros_like(X, dtype=float)
    for j, d in enumerate(degrees):
        if d == 0:
            Z[:, j] = np.log(X[:, j])
        else:
            Z[:, j] = X[:, j] ** d
    return Z


In [7]:
def linear_predictor(X, beta, degrees):
    return fractional_polynomial(X, degrees) @ beta


In [8]:
def weibull_cumulative_hazard(t, shape, scale):
    return (t / scale) ** shape

In [9]:
def survival_fp(t, X, beta, degrees, shape, scale):
    """
    Returns S(t | x) for all samples
    """
    eta = linear_predictor(X, beta, degrees)
    H0 = weibull_cumulative_hazard(t, shape, scale)
    return np.exp(-H0 * np.exp(eta))

In [10]:
def fit_censoring_km(T, E):
    """
    T: event/censoring times
    E: event indicator (1=event, 0=censored)
    """
    km = KaplanMeierFitter()
    km.fit(T, event_observed=1 - E)
    return km


In [11]:
def brier_score_fp(
    t, X, T, E,
    beta, degrees,
    shape, scale,
    km_censor
):
    S_hat = survival_fp(t, X, beta, degrees, shape, scale)
    Y = (T > t).astype(float)

    G_t = km_censor.predict(t)
    G_T = np.array([km_censor.predict(min(ti, t)) for ti in T])

    weights = np.where(
        (T <= t) & (E == 1),
        1.0 / np.maximum(G_T, 1e-8),
        1.0 / np.maximum(G_t, 1e-8)
    )

    return np.mean(weights * (Y - S_hat) ** 2)

In [12]:
def integrated_brier_score_fp(
    X, T, E,
    beta, degrees,
    shape, scale,
    t_max=None,
    n_times=100
):
    if t_max is None:
        t_max = np.max(T)

    times = np.linspace(0.01, t_max, n_times)
    km_censor = fit_censoring_km(T, E)

    bs = [
        brier_score_fp(
            t, X, T, E,
            beta, degrees,
            shape, scale,
            km_censor
        )
        for t in times
    ]

    return np.trapz(bs, times) / t_max


Initialize population

Chromosome structure

Let:

* first p genes = β coefficients

* next p genes = FP degrees (continuous, later discretized if needed)

chromosome = [β₁,…,βₚ, d₁,…,dₚ]

In [13]:
def initialize_population(pop_size, num_parameters):
    return np.random.uniform(-4, 4, size=(pop_size, num_parameters))

Crossover Functions

In [14]:
# Single-point crossover
def crossover(parent1, parent2):
    if random.random() < CROSSOVER_RATE:
        point = random.randint(1, len(parent1) - 1)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2
    return parent1.copy(), parent2.copy()

# Two-point crossover
def two_point_crossover(parent1, parent2):

    if random.random() > CROSSOVER_RATE:
        return parent1.copy(), parent2.copy()
    
    length = len(parent1)
    point1 = random.randint(0, length - 2)
    point2 = random.randint(point1 + 1, length - 1)

    offspring1 = np.concatenate([parent1[:point1], parent2[point1:point2], parent1[point2:]])
    offspring2 = np.concatenate([parent2[:point1], parent1[point1:point2], parent2[point2:]])

    return offspring1, offspring2

# Uniform crossover
def uniform_crossover(parent1, parent2, crossover_prob=0.5):
    """
    Performs uniform crossover on two parent chromosomes.

    For each gene, a random decision is made whether to swap the genes
    from the parents, based on the crossover_prob.

    Args:
        parent1 (list): The first parent chromosome.
        parent2 (list): The second parent chromosome.
                        Must be the same length as parent1.
        crossover_prob (float): The probability of swapping a gene from
                                parent1 to parent2. Must be between 0.0 and 1.0.

    Returns:
        tuple: A tuple containing two new offspring chromosomes.
    """
    if len(parent1) != len(parent2):
        raise ValueError("Parents must have the same length for crossover.")

    length = len(parent1)
    offspring1 = [None] * length
    offspring2 = [None] * length

    for i in range(length):
        if random.random() < crossover_prob:
            offspring1[i] = parent2[i]
            offspring2[i] = parent1[i]
        else:
            offspring1[i] = parent1[i]
            offspring2[i] = parent2[i]

    return offspring1, offspring2

Mutation - Gaussian perturbation

In [ ]:
def mutate(chromosome, sigma=0.1):
    for i in range(len(chromosome)):
        if random.random() < MUTATION_RATE:
            chromosome[i] += np.random.normal(0, sigma)
    return chromosome

Tournament Selection Function

In [16]:

def tournament_selection(population, fitnesses, tournament_size=3):
    indices = np.random.choice(len(population), tournament_size)
    best_idx = indices[np.argmax([fitnesses[i] for i in indices])]
    return population[best_idx]

Fitness evaluation wrapper

* GA maximizes fitness

* IBS should be minimized

So we use: fitness = - IBS

In [17]:
def evaluate_fitness(
    chromosome, X, T, E,
    p, shape, scale
):
    beta = chromosome[:p]
    degrees = chromosome[p:]

    ibs = integrated_brier_score_fp(
        X, T, E,
        beta=beta,
        degrees=degrees,
        shape=shape,
        scale=scale
    )

    return -ibs

baseline hazard estimate using Breslow estimator

In [18]:
def breslow_cumulative_hazard(T, E, eta):
    order = np.argsort(T)
    T, E, eta = T[order], E[order], eta[order]

    H0 = np.zeros_like(T, dtype=float)
    risk = np.exp(eta)

    for i in range(len(T)):
        if E[i] == 1:
            H0[i:] += 1.0 / np.sum(risk[i:])

    return T, H0


##### Main GA Algorithm

In [19]:
def run_ga(
    X, T, E,
    p,
    shape, scale,
    pop_size=30,
    generations=50,
    crossover_type="two_point"
):
    num_parameters = 2 * p

    # Initialize population
    population = initialize_population(pop_size, num_parameters)

    best_fitness_history = []
    best_solution = None
    best_fitness = -np.inf

    for gen in range(generations):
        # Evaluate fitness
        fitnesses = np.array([
            evaluate_fitness(ind, X, T, E, p, shape, scale)
            for ind in population
        ])

        # Track best
        gen_best_idx = np.argmax(fitnesses)
        if fitnesses[gen_best_idx] > best_fitness:
            best_fitness = fitnesses[gen_best_idx]
            best_solution = population[gen_best_idx].copy()

        best_fitness_history.append(best_fitness)

        # New population
        new_population = []

        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)

            if crossover_type == "single":
                child1, child2 = crossover(parent1, parent2)
            elif crossover_type == "uniform":
                child1, child2 = uniform_crossover(parent1, parent2)
            else:
                child1, child2 = two_point_crossover(parent1, parent2)

            child1 = mutate(child1)
            child2 = mutate(child2)

            new_population.append(child1)
            if len(new_population) < pop_size:
                new_population.append(child2)

        population = np.array(new_population)

        if gen % 5 == 0 or gen == generations - 1:
            print(
                f"Gen {gen:3d} | "
                f"Best IBS: {-best_fitness:.4f}"
            )

    return best_solution, best_fitness, best_fitness_history


Experiment

In [20]:
def run_experiment(
    X, T, E,
    shape=1.5,
    scale=10.0
):
    """
    Runs GA experiments for ONE dataset across multiple configs
    """
    results = []

    p = X.shape[1]

    for cfg in configs:
        print(f"\nRunning config {cfg['id']}")

        best_sol, best_fit, history = run_ga(
            X, T, E,
            p=p,
            shape=shape,
            scale=scale,
            pop_size=cfg["pop_size"],
            generations=cfg["generation"],
            crossover_type="two_point"
        )

        beta = best_sol[:p]
        degrees = best_sol[p:]

        results.append({
            "config_id": cfg["id"],
            "pop_size": cfg["pop_size"],
            "generations": cfg["generation"],
            "best_ibs": -best_fit,
            "beta": beta,
            "degrees": degrees,
            "history": history
        })

    return results


GBSG

In [21]:
X,T,E = load_dataset(path_gbsg, "rfstime", "status")

In [22]:
result_gbsg = run_experiment(X,T,E)


Running config 1


C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))

Gen   0 | Best IBS: inf


C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))
C:\Users\HP\AppData\Local\Temp\ipykernel_16204\1234694779.py:7: RuntimeWarning: overflow encountered in exp
  return np.exp(-H0 * np.exp(eta))

KeyboardInterrupt: 